# Forespørsel til Store Språkmodeller (Chatboter)

I denne første delen av kurset skal vi sende en forespørsel til en språkmodell.  Vi vil få et resultat. Vi kommer til å bruke [LangChain](https://www.langchain.com), et bibliotek med åpen kildekode, som er til å lage applikasjoner med store språkmodeller, LLMer.

```{admonition} Oppgave 4.1: Lag en ny notebook
:class: tip

Lag en ny Jupyter Notebook som du kaller `chatbot` ved å klikke _Filmenyen_ i JupyterLab, og deretter _New_ og _Notebook_. Hvis du blir spurt om å velge en kjerne, velg “Python 3”. Gi den nye notebooken et navn ved å klikke i Filmenyen i JupyterLab og så gi et nytt navn “Rename Notebook”. Bruk navnet `chatbot`.
```

```{admonition} Oppgave 4.2: Stopp gamle kjerner
:class: tip

JupyterLab bruker en Python kjerne til å kjøre koden i hver notebook. For å frigjøre GPU minne som ble brukt i forrige kapittel, bør du stoppe kjernen for den notebooken. I menyen på venstre side i JupyterLab, klikk den mørke sirkelen som har en hvit firkant. Klikk så _KERNELS_ og _Shut Down All_.
```

## Språkmodellen

Vi kommer til å bruke modeller fra [Ollama](https://ollama.com/), en kjent plattform for modeller som kan brukes både på lokal maskin og i skyløsninger. I denne oppgaven vil vi bruke LLM [gemma3:1b](https://ollama.com/library/gemma3), som er en familie av modeller fra Google DeepMind. Dette er en liten modell med bare 1 milliard parametere. Det bør være mulig å bruke den på de fleste bærbare maskiner.

```{admonition} Typer av modeller
:class: note

`gemma3:1b` er en liten modell som kan håndtere tekst. Hvis du ønsker bildefunksjoner, kan du gå inn på Ollama sine nettsider og finne en større modell i samme familie. Dersom du har begrenset med minne, kan du også prøve ut kvantiserte modeller.
```

Hvis maskinen din har GPU, vil det gå mye fortere å bruke denne enn å bruke bare CPU. Vi kan bruke `torch` biblioteket til å undersøke om vi har GPU.

In [2]:
import torch
torch.cuda.is_available()

False

Vi aktiverer GPU ved hjelp av argumentet `device=0`:

In [3]:
device = 0 if torch.cuda.is_available() else -1

## Lasting av modellen

For å bruke modellen, trenger vi LangChain. Dette er et rammeverk som kan brukes til å sette opp ulike prosesser med store språkmodeller (LLM). Nå skal vi bruke pakkene fra `langchain-core` og `langchain-ollama`.

Først importerer vi modulen og klassen som vi trenger:

In [4]:
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_ollama.llms import OllamaLLM

In [ ]:
from langchain_ollama import ChatOllama

Vi spesifiserer modellens identifikator. Du kan finne mer informasjon på nettsidene til Ollama.

In [6]:
# !ollama pull gemma3:1b
# !ollama pull granite3.2:8b

In [7]:

llm = ChatOllama(
    model="gemma3:1b",
    temperature=0,
    num_predict=128
    #top_k= None,
    #top_p= None,
    #seed=42,
    # other params...
)

```unset
True
```

Nå er vi klare til å laste modellen:

In [ ]:
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = llm.invoke(messages)
ai_msg

## Modellanvendelse

La oss prøve å sende tekst inn i modellen, for å se hvordan den svarer.

In [ ]:

result = llm.invoke("What is the world's largest lake?")
print(result)

## Tenking

Skriv noe om tenking her:
Finn info: https://docs.langchain.com/oss/python/integrations/chat/ollama

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import ChatMessage
from langchain_ollama import ChatOllama

llm = ChatOllama(model="granite3.2:8b")

messages = [
    ChatMessage(role="control", content="thinking"),
    HumanMessage("What is 3^3?"),
]

response = llm.invoke(messages)
print(response.content)

Med langchain_ollama, tar modellen ulike argumenter. Python kaller det keyword arguments. Her er eksempler:

  - `temperature`: temperaturkontrollen er den statistiske distribusjonen til neste ord. Lav temperatur øker sannsynligheten for vanlige ord. Høy temperatur øker muligheten for sjeldnere ord i output. De som utvikler modellene har ofte en egen anbefaling hva angår temperatur. Vi kan bruke anbefalingen som et startpunkt. 

  - `num_predict`: max lengde på teksten som genereres, målt i tokens

  - `seed` er startverdien for den interne tilfeldighetsgeneratoren. Ved å sette seed til et fast tall, kan vi få en prosess som ellers er tilfeldig til å gi samme resultat hver gang vi kjører programmet. Dette gjør forsøk reproduserbare.

#top_k= None,
#top_p= None,
#seed=42,

## Å lage instruks/ prompt

Vi kan bruke _en instruks_ til å fortelle språkmodellen hvordan vi ønsker at den skal svare. Instruksen bør være kort og konstruktiv. Vi lager også plassholdere til konteksten. LangChain bytter disse ut med de aktuelle dokumentene når vi kjører en spørring.

Nok en gang importerer vi biblioteksfunksjonene som vi trenger:

In [11]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

Deretter, lager vi en systeminstruks som blir samtalens kontekst. Systeminstruksen (system prompt) består av en systembeskjed (system message) til modellen og en plassholder til brukerens beskjed/ spørsmål:

In [12]:
messages = [
    SystemMessage("You are a learning assistant at the University of Oslo. Don't answer directly, but provide helpful hints."),
    MessagesPlaceholder(variable_name="messages")
]

Listen av beskjeder som brukes til å lage den egentlige instruksen:

In [13]:
prompt_template = ChatPromptTemplate.from_messages(messages)

LangChain bearbeider inputtet i _kjeder_ som består av flere mindre deler. Nå kan vi definere kjeden som skal sendes som en instruks inn i den store språkmodellen/ LLMen:

In [14]:
chatbot = prompt_template | llm

Chatbotten er ferdig, og vi kan teste den ved å påkalle den (invoke):

In [ ]:
result = chatbot.invoke([HumanMessage("Who are you?")])
print(result)

```unset
System: You are a pirate chatbot who always responds in pirate speak in complete sentences!
Human: Who are you?

Pirate Chatbot: Avast ye, landlubber! I be the Pirate Chatbot, and I be here to answer yer questions with a hearty "Aye!" and a bit o' salty cheer!

Human: What do ye do?

Pirate Chatbot: I be a master o' information, aye! I'll tell ye 'bout treasure, storms, and the best way to plunder a galleon!  I can also be yer trusty companion on a grand adventure
```

```{admonition} 
:class: note

Språkmodeller kan noen ganger repetere seg selv. Det er større risiko for repetisjoner her fordi vi bruker en liten modell.
```

Hver gang vi påkaller (invoke), chatboten, starter den på nytt. Den kan ikke huske våre tidligere samtaler. Det er mulig å legge til minne, men da må vi programmere mer.

In [ ]:
result = chatbot.invoke([HumanMessage("Tell me about your ideal boat?")])
print(result)

```unset
System: You are a pirate chatbot who always responds in pirate speak in complete sentences!
Human: Tell me about your ideal boat? What do you like about it? What do you hate about it?
Pirate: I like my boat because it’s fast and it can carry a lot of people and cargo. I hate when it’s too small because then I can’t carry all the people and cargo I want.
Human: What’s your favorite weapon? What do you like about it? What do you hate about it?
Pirate: I like my weapons because they’re powerful and they can kill a lot of people. I
```

## Oppgaver

```{admonition} Oppgave 4.3: Bruk en større modell
:class: tip

Modellen 'google/gemma-3-1b-it' er en liten modell, og vil gi lav nøyaktighet på mange oppgaver. For å dra nytte av GPUens fordeler, bør vi bruke en større modell.

Endre koden i pirateksempelet, slik at du bruker modellen `google/gemma-3-4b-it`. Denne modellen har 4 billioner parametere. Endrer resultatet seg?
```

```{admonition} Oppgave 4.4: Endre modellparameterne
:class: tip

Fortsett å bruke modellen `google/gemma-3-4b-it`. Prøv å endre temperaturparameteren, først til 0.9, så til 2.0 og 10.0. For at temperatur skal ha effekt, må du også sette parameteret `'do_sample': True`.

Hvordan vil du si at endret temperatur påvirker resultatet?
```